# AutoETS - Automatic Exponential Smoothing

This notebook demonstrates **automatic ETS model selection** using `forecastbox.auto.AutoETS`.

## ETS Framework

ETS stands for **Error, Trend, Seasonality** — a taxonomy that classifies exponential smoothing
models by their three components:

- **Error**: how the noise enters the model — **A**dditive or **M**ultiplicative
- **Trend**: the long-term direction — **N**one, **A**dditive, **A**dditive **d**amped, **M**ultiplicative, **M**ultiplicative **d**amped
- **Seasonal**: repeating patterns — **N**one, **A**dditive, **M**ultiplicative

This gives $2 \times 5 \times 3 = 30$ possible model specifications. AutoETS fits all
admissible combinations and selects the one with the best information criterion (AIC, AICc, or BIC).

**Topics covered:**
- The complete ETS taxonomy (30 models)
- Automatic model selection with AutoETS
- Additive vs multiplicative components
- Damped trend models
- Application to M3 competition series

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecastbox.auto import AutoETS

import sys
sys.path.insert(0, "..")
from utils.helpers import load_airline, load_m3_sample, get_series

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)

## 1. ETS Taxonomy

The full taxonomy of 30 ETS models:

| | **Seasonal: N** | **Seasonal: A** | **Seasonal: M** |
|---|---|---|---|
| **Error: A, Trend: N** | ETS(A,N,N) — SES | ETS(A,N,A) | ETS(A,N,M) |
| **Error: A, Trend: A** | ETS(A,A,N) — Holt | ETS(A,A,A) — Holt-Winters Add. | ETS(A,A,M) |
| **Error: A, Trend: Ad** | ETS(A,Ad,N) — Damped | ETS(A,Ad,A) | ETS(A,Ad,M) |
| **Error: A, Trend: M** | ETS(A,M,N) | ETS(A,M,A) | ETS(A,M,M) |
| **Error: A, Trend: Md** | ETS(A,Md,N) | ETS(A,Md,A) | ETS(A,Md,M) |
| **Error: M, Trend: N** | ETS(M,N,N) | ETS(M,N,A) | ETS(M,N,M) |
| **Error: M, Trend: A** | ETS(M,A,N) | ETS(M,A,A) | ETS(M,A,M) — Holt-Winters Mult. |
| **Error: M, Trend: Ad** | ETS(M,Ad,N) | ETS(M,Ad,A) | ETS(M,Ad,M) |
| **Error: M, Trend: M** | ETS(M,M,N) | ETS(M,M,A) | ETS(M,M,M) |
| **Error: M, Trend: Md** | ETS(M,Md,N) | ETS(M,Md,A) | ETS(M,Md,M) |

**Common models:**
- **ETS(A,N,N)**: Simple Exponential Smoothing (SES) — level only
- **ETS(A,A,N)**: Holt's linear method — level + trend
- **ETS(A,A,A)**: Holt-Winters additive — level + trend + additive seasonality
- **ETS(M,A,M)**: Holt-Winters multiplicative — level + trend + multiplicative seasonality
- **ETS(M,Ad,M)**: Damped multiplicative — most common for real-world seasonal data

Let's load the airline data and visualize its seasonal decomposition:

In [ ]:
# Load airline data
df_airline = load_airline()
airline = df_airline["passengers"]

# Seasonal decomposition to understand the components
from statsmodels.tsa.seasonal import seasonal_decompose

decomp = seasonal_decompose(airline, model="multiplicative", period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
decomp.observed.plot(ax=axes[0], color="steelblue")
axes[0].set_title("Observed")
axes[0].set_ylabel("Passengers")

decomp.trend.plot(ax=axes[1], color="darkorange")
axes[1].set_title("Trend")
axes[1].set_ylabel("Trend")

decomp.seasonal.plot(ax=axes[2], color="forestgreen")
axes[2].set_title("Seasonal (multiplicative)")
axes[2].set_ylabel("Seasonal Factor")

decomp.resid.plot(ax=axes[3], color="firebrick")
axes[3].set_title("Residual")
axes[3].set_ylabel("Residual")

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The seasonal swings grow with the level → multiplicative seasonality is appropriate.")

## 2. AutoETS Selection

AutoETS fits all admissible ETS model combinations and selects the best by information criterion.
The algorithm:

1. Enumerate all valid (Error, Trend, Seasonal) combinations
2. Apply admissibility restrictions (some multiplicative combinations can be unstable)
3. Fit each model via maximum likelihood
4. Select the model with the lowest AICc (or AIC/BIC)

In [ ]:
# Fit AutoETS to airline data (seasonal_period=12 for monthly data)
auto_ets = AutoETS(seasonal_period=12, ic="aicc")
result = auto_ets.fit(airline)

print("Selected model:", result.model_type)
print(f"  Error:    {result.error}")
print(f"  Trend:    {result.trend}")
print(f"  Seasonal: {result.seasonal}")
print(f"  Damped:   {result.damped}")
print(f"  AICc:     {result.ic_value:.2f}")
print(f"  Models evaluated: {result.n_fits}")

print("\n" + result.summary())

# Show top 5 models considered
if result.all_models is not None:
    print("\nTop 5 ETS models by AICc:")
    print(result.all_models.head(5).to_string())

## 3. Comparing ETS Variants

A key choice in ETS is whether to use **additive** or **multiplicative** components:

- **Additive seasonality**: seasonal effect is constant in absolute terms (e.g., always +20 in summer)
- **Multiplicative seasonality**: seasonal effect scales with the level (e.g., always +15% in summer)

For airline data, where seasonal amplitude grows with the level, multiplicative is expected to
perform better. Let's verify by forcing each variant:

In [ ]:
# Force additive model: ETS(A,A,A)
ets_additive = AutoETS(seasonal_period=12, error="A", trend="A", seasonal="A")
res_add = ets_additive.fit(airline)

# Force multiplicative model: ETS(M,A,M)
ets_multiplicative = AutoETS(seasonal_period=12, error="M", trend="A", seasonal="M")
res_mul = ets_multiplicative.fit(airline)

print(f"Additive ETS(A,A,A):")
print(f"  AICc: {res_add.ic_value:.2f}")

print(f"\nMultiplicative ETS(M,A,M):")
print(f"  AICc: {res_mul.ic_value:.2f}")

diff = res_add.ic_value - res_mul.ic_value
print(f"\nDifference (Add - Mult): {diff:.2f}")
if diff > 0:
    print("→ Multiplicative model is preferred (lower AICc)")
else:
    print("→ Additive model is preferred (lower AICc)")

# Compare forecasts visually
fc_add = res_add.forecast(h=24)
fc_mul = res_mul.forecast(h=24)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fc_index = pd.date_range(start=airline.index[-1] + pd.DateOffset(months=1), periods=24, freq="MS")

for ax, fc, title in [(axes[0], fc_add, f"ETS(A,A,A) — AICc={res_add.ic_value:.1f}"),
                       (axes[1], fc_mul, f"ETS(M,A,M) — AICc={res_mul.ic_value:.1f}")]:
    ax.plot(airline.index, airline.values, color="steelblue", linewidth=1.5, label="Historical")
    ax.plot(fc_index, fc.point, color="darkorange", linewidth=2, label="Forecast")
    if fc.lower_95 is not None and fc.upper_95 is not None:
        ax.fill_between(fc_index, fc.lower_95, fc.upper_95, alpha=0.2, color="darkorange")
    ax.set_title(title)
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Damped Trend

The **damped trend** modification (Gardner & McKenzie, 1985) adds a damping parameter $\phi \in (0, 1)$
that gradually flattens the trend as the forecast horizon increases:

$$\hat{y}_{T+h} = \ell_T + (\phi + \phi^2 + \cdots + \phi^h) b_T$$

As $h \to \infty$, the forecast converges to $\ell_T + \frac{\phi}{1-\phi} b_T$ (a finite asymptote).

**When to use damped trends:**
- For long-horizon forecasts where indefinite growth is unrealistic
- When you're uncertain whether the trend will persist
- In competitions: damped models consistently outperform undamped models at longer horizons

In [ ]:
# Compare damped vs undamped trend on airline data
# Undamped: ETS(M,A,M)
ets_undamped = AutoETS(seasonal_period=12, error="M", trend="A", seasonal="M", damped=False)
res_undamped = ets_undamped.fit(airline)

# Damped: ETS(M,Ad,M)
ets_damped = AutoETS(seasonal_period=12, error="M", trend="Ad", seasonal="M", damped=True)
res_damped = ets_damped.fit(airline)

print(f"Undamped ETS(M,A,M):  AICc = {res_undamped.ic_value:.2f}")
print(f"Damped   ETS(M,Ad,M): AICc = {res_damped.ic_value:.2f}")

# Forecast 48 months ahead to see the divergence
h = 48
fc_undamped = res_undamped.forecast(h=h)
fc_damped = res_damped.forecast(h=h)

fc_index = pd.date_range(start=airline.index[-1] + pd.DateOffset(months=1), periods=h, freq="MS")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(airline.index, airline.values, color="steelblue", linewidth=1.5, label="Historical")
ax.plot(fc_index, fc_undamped.point, color="darkorange", linewidth=2, label="Undamped (A trend)")
ax.plot(fc_index, fc_damped.point, color="forestgreen", linewidth=2, linestyle="--", label="Damped (Ad trend)")
ax.set_title("Damped vs Undamped Trend — 48-Month Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Passengers (thousands)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nNote how the damped forecast grows more slowly — the trend gradually levels off.")

## 5. AutoETS on M3 Series

Let's apply AutoETS to the M3 sample dataset and see which model type is selected for each series.
This demonstrates how AutoETS adapts to different data characteristics.

In [ ]:
# Load M3 sample
m3 = load_m3_sample()
all_ids = m3["series_id"].unique()

# Determine seasonal period by frequency
freq_map = {"monthly": 12, "quarterly": 4}

results_table = []
for sid in all_ids:
    y = get_series(m3, sid)
    freq = m3[m3["series_id"] == sid]["frequency"].iloc[0]
    sp = freq_map.get(freq, 1)

    auto = AutoETS(seasonal_period=sp, ic="aicc")
    res = auto.fit(y)
    results_table.append({
        "Series": sid,
        "Frequency": freq,
        "Model": res.model_type,
        "Error": res.error,
        "Trend": res.trend,
        "Seasonal": res.seasonal,
        "Damped": res.damped,
        "AICc": round(res.ic_value, 2),
    })

ets_results = pd.DataFrame(results_table)
print("AutoETS Results for M3 Sample Series:")
print(ets_results.to_string(index=False))

## Exercise 1: Which ETS model is best for quarterly_stationary?

Load the `quarterly_stationary` series from the M3 sample. Fit AutoETS with `seasonal_period=4`.
What model does AutoETS select? Does the result make sense for a stationary series (hint: expect
no trend and possibly no seasonality)?

In [ ]:
# TODO: Exercise 1 — Find the best ETS model for quarterly_stationary
# Hints:
# 1. y = get_series(m3, "quarterly_stationary")
# 2. Fit AutoETS with seasonal_period=4
# 3. Print the selected model type and AICc
# 4. Does the model have trend or seasonality components?

## Exercise 2: Compare AutoETS forecast accuracy vs AutoARIMA on airline

Fit both AutoETS and AutoARIMA on the airline dataset. Compare their forecasts for h=24 on a
single plot. Which model produces tighter prediction intervals?

In [ ]:
# TODO: Exercise 2 — Compare AutoETS vs AutoARIMA on airline
# Hints:
# 1. from forecastbox.auto import AutoARIMA
# 2. Fit AutoARIMA(seasonal=True, m=12) and AutoETS(seasonal_period=12) on airline
# 3. Forecast h=24 from each
# 4. Plot both forecasts on the same axes with prediction intervals
# 5. Compare: AICc values, forecast shapes, interval widths